In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings(action='ignore')

In [2]:
import os
# 노트북 파일이 있는 폴더로 이동 (예시)
os.chdir(r'C:\githome\hipython_rep')

# 변경 후 확인
print("변경 후:", os.getcwd())

변경 후: c:\githome\hipython_rep


In [38]:
X = pd.read_csv('./data/X.csv', encoding='euc-kr')
y = pd.read_csv('./data/y.csv', encoding='euc-kr')

In [39]:
# cust_id 기준으로 병합
df = pd.merge(X, y, on='cust_id', how='inner')


In [40]:
print(df.head())
print(df.shape)


   cust_id      총구매액     최대구매액       환불금액   주구매상품 주구매지점  내점일수   내점당구매건수  \
0        0  68282840  11264000  6860000.0      기타   강남점    19  3.894737   
1        1   2136000   2136000   300000.0     스포츠   잠실점     2  1.500000   
2        2   3197000   1639000        NaN  남성 캐주얼   관악점     2  2.000000   
3        3  16077620   4935000        NaN      기타   광주점    18  2.444444   
4        4  29050000  24000000        NaN      보석  본  점     2  1.500000   

     주말방문비율  구매주기  gender  
0  0.527027    17       0  
1  0.000000     1       0  
2  0.000000     1       1  
3  0.318182    16       1  
4  0.000000    85       0  
(3500, 11)


In [41]:
df

,cust_id,총구매액,최대구매액,환불금액,주구매상품,주구매지점,내점일수,내점당구매건수,주말방문비율,구매주기,gender
0,0,68282840,11264000,6860000.0,기타,강남점,19,3.894737,0.527027,17,0
1,1,2136000,2136000,300000.0,스포츠,잠실점,2,1.500000,0.000000,1,0
2,2,3197000,1639000,NaN,남성 캐주얼,관악점,2,2.000000,0.000000,1,1
3,3,16077620,4935000,NaN,기타,광주점,18,2.444444,0.318182,16,1
4,4,29050000,24000000,NaN,보석,본 점,2,1.500000,0.000000,85,0
...,...,...,...,...,...,...,...,...,...,...,...
3495,3495,3175200,3042900,NaN,골프,본 점,1,2.000000,1.000000,0,1
3496,3496,29628600,7200000,6049600.0,시티웨어,부산본점,8,1.625000,0.461538,40,1
3497,3497,75000,75000,NaN,주방용품,창원점,1,1.000000,0.000000,0,0
3498,3498,1875000,1000000,NaN,화장품,본 점,2,1.000000,0.000000,39,0


In [42]:
df['환불금액'] = np.where(
    (df['총구매액'] == 0) & (df['환불금액'].isna()),
    0,
    df['환불금액']
)

In [43]:
# 1. 환불금액이 NaN이 아닌 행만 사용해 환불률 계산
valid_refund_df = df[df['환불금액'].notna()].copy()

# 총구매액이 0인 경우는 제외 (inf 방지)
valid_refund_df = valid_refund_df[valid_refund_df['총구매액'] != 0]

# 2. 성별별 평균 환불률 계산
valid_refund_df['환불률'] = valid_refund_df['환불금액'] / valid_refund_df['총구매액']
avg_refund_rate_by_gender = valid_refund_df.groupby('gender')['환불률'].mean()

# 3. NaN 환불금액을 성별 평균 환불률로 대체
def fill_missing_refund(row):
    if pd.isna(row['환불금액']):
        refund_rate = avg_refund_rate_by_gender.loc[row['gender']]
        return row['총구매액'] * refund_rate
    else:
        return row['환불금액']

df['환불금액'] = df.apply(fill_missing_refund, axis=1)


In [44]:
df

,cust_id,총구매액,최대구매액,환불금액,주구매상품,주구매지점,내점일수,내점당구매건수,주말방문비율,구매주기,gender
0,0,68282840,11264000,6.860000e+06,기타,강남점,19,3.894737,0.527027,17,0
1,1,2136000,2136000,3.000000e+05,스포츠,잠실점,2,1.500000,0.000000,1,0
2,2,3197000,1639000,1.390199e+06,남성 캐주얼,관악점,2,2.000000,0.000000,1,1
3,3,16077620,4935000,6.991269e+06,기타,광주점,18,2.444444,0.318182,16,1
4,4,29050000,24000000,2.519406e+07,보석,본 점,2,1.500000,0.000000,85,0
...,...,...,...,...,...,...,...,...,...,...,...
3495,3495,3175200,3042900,1.380719e+06,골프,본 점,1,2.000000,1.000000,0,1
3496,3496,29628600,7200000,6.049600e+06,시티웨어,부산본점,8,1.625000,0.461538,40,1
3497,3497,75000,75000,6.504489e+04,주방용품,창원점,1,1.000000,0.000000,0,0
3498,3498,1875000,1000000,1.626122e+06,화장품,본 점,2,1.000000,0.000000,39,0


In [45]:
# y: 성별(예측 대상)
y = df['gender']

# X: 성별 컬럼 제외한 나머지
X = df.drop(columns=['gender'])


In [46]:
X

,cust_id,총구매액,최대구매액,환불금액,주구매상품,주구매지점,내점일수,내점당구매건수,주말방문비율,구매주기
0,0,68282840,11264000,6.860000e+06,기타,강남점,19,3.894737,0.527027,17
1,1,2136000,2136000,3.000000e+05,스포츠,잠실점,2,1.500000,0.000000,1
2,2,3197000,1639000,1.390199e+06,남성 캐주얼,관악점,2,2.000000,0.000000,1
3,3,16077620,4935000,6.991269e+06,기타,광주점,18,2.444444,0.318182,16
4,4,29050000,24000000,2.519406e+07,보석,본 점,2,1.500000,0.000000,85
...,...,...,...,...,...,...,...,...,...,...
3495,3495,3175200,3042900,1.380719e+06,골프,본 점,1,2.000000,1.000000,0
3496,3496,29628600,7200000,6.049600e+06,시티웨어,부산본점,8,1.625000,0.461538,40
3497,3497,75000,75000,6.504489e+04,주방용품,창원점,1,1.000000,0.000000,0
3498,3498,1875000,1000000,1.626122e+06,화장품,본 점,2,1.000000,0.000000,39


0       0
1       0
2       1
3       1
4       0
       ..
3495    1
3496    1
3497    0
3498    0
3499    0
Name: gender, Length: 3500, dtype: int64